# Peru Solar Station Processing Tutorial

This notebook processes Peruvian station observations and creates the monthly station climatology used by the ATLAS solar workflow.

The final output is:

```text
../data/stations/peru/allstats_solar_radiation.csv
```

The output table contains one row per station and month with the following fields:

```text
id, month, ssrd, name, latitude, longitude, altitude
```

Users should only edit the **User parameters** cell, then run the notebook from top to bottom.

## Step 1. User parameters

Set the country and the input file names.

All input files are expected inside:

```text
../data/stations/{country}/
```

For Peru, this notebook supports two station data sources:

1. **DGIA daily radiation and sunshine files**, stored in a folder containing multiple Excel files.
2. **DAVIS/PYTO monthly summary file**, with a station coordinate table.

In [1]:
from pathlib import Path

country = "peru"

# Main input and output folder.
base_path = Path(f"../data/stations/{country}")
output_path = base_path
output_path.mkdir(parents=True, exist_ok=True)

# Source 1: DGIA station metadata and Excel folder.
station_info_path = base_path / "stations_info.xlsx"
source1_folder = base_path / "DATOS RAD_SOLAR_Y_HELIOFANIA_DGIA"

# Source 2: DAVIS/PYTO station metadata and monthly summary Excel file.
source2_folder = base_path / "DATOS_RAD_SOLAR_DAVIS_PYTO_MEM-SENAMHI-DGIA_PARA_OGEI"
source2_station_coordinates_path = source2_folder / "stations_coordinates_decimal.csv"
source2_summary_path = source2_folder / "resumen_estacciones_peru.xlsx"

# Final output.
output_csv = output_path / "allstats_solar_radiation.csv"

# Optional filters used in the original workflow to avoid duplicated or problematic stations.
excluded_station_names = ["MIRAFLORES", "San Camilo"]

# Conversion used by the original notebook:
# MJ m-2 day-1 to W m-2.
mj_m2_day_to_w_m2 = 11.574

# Conversion from W m-2 to kWh m-2 day-1 used by the downstream workflow.
w_m2_to_kwh_m2_day = 24 / 1000

## Step 2. Imports

In [2]:
import pandas as pd
import numpy as np

## Step 3. Helper functions

These functions clean station names, remove outliers and standardise the final output columns.

In [3]:
def normalize_name(value):
    """Normalise station names for matching tables."""
    return str(value).strip().upper().replace(" ", "_")


def remove_outliers_iqr(df, columns):
    """Remove outliers using the interquartile range method."""
    df_clean = df.copy()

    for col in columns:
        if col not in df_clean.columns:
            continue

        values = pd.to_numeric(df_clean[col], errors="coerce")
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        if pd.isna(iqr) or iqr == 0:
            continue

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        df_clean = df_clean[(values >= lower) & (values <= upper)]

    return df_clean


def extract_station_name(path):
    """Extract the station name from the DGIA Excel file name."""
    filename = Path(path).name
    parts = filename.replace(".xlsx", "").replace(".xls", "").split("_")
    return "_".join(parts[3:-1])


def standardise_output(df):
    """Return the common ATLAS station table layout."""
    expected_columns = ["id", "month", "ssrd", "name", "latitude", "longitude", "altitude"]

    for col in expected_columns:
        if col not in df.columns:
            df[col] = pd.NA

    df = df.loc[:, expected_columns].copy()
    df["month"] = pd.to_numeric(df["month"], errors="coerce").astype("Int64")
    df["ssrd"] = pd.to_numeric(df["ssrd"], errors="coerce")
    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df["altitude"] = pd.to_numeric(df["altitude"], errors="coerce")

    return df.sort_values(["name", "month"]).reset_index(drop=True)

## Step 4. Source 1 processing functions

In [4]:
def process_dgia_daily_file(df, mj_m2_day_to_w_m2=11.574):
    """Clean one DGIA daily Excel table."""
    df = df.copy()

    # Some files contain a duplicated header in the first row.
    if len(df) > 0:
        df = df.drop(index=0).reset_index(drop=True)

    df = df.rename(
        columns={
            "AÑO": "year",
            "ANIO": "year",
            "MES": "month",
            "DIA": "day",
            "RSOLAR": "solar_radiation_MJ_m2_day",
            "RAD.SOL": "solar_radiation_MJ_m2_day",
            "HSOL": "sunshine_hours",
        }
    )

    required = ["year", "month", "day"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing date columns: {missing}")

    df["time"] = pd.to_datetime(df[["year", "month", "day"]], errors="coerce")
    df = df.dropna(subset=["time"]).set_index("time")

    if "solar_radiation_MJ_m2_day" in df.columns:
        df["solar_radiation_W_m2"] = (
            pd.to_numeric(df["solar_radiation_MJ_m2_day"], errors="coerce") * mj_m2_day_to_w_m2
        )

    if "sunshine_hours" in df.columns:
        df["sunshine_hours"] = pd.to_numeric(df["sunshine_hours"], errors="coerce")

    # Remove columns with too many missing values.
    nan_percent = df.isna().mean() * 100
    df = df.loc[:, nan_percent < 70]

    df = remove_outliers_iqr(df, ["solar_radiation_W_m2", "sunshine_hours"])

    columns_to_drop = ["year", "month", "day", "solar_radiation_MJ_m2_day"]
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

    return df


def process_dgia_folder(input_folder, station_info_path):
    """Process all DGIA Excel files and return one cleaned table."""
    input_folder = Path(input_folder)
    station_info = pd.read_excel(station_info_path)

    excel_files = sorted(list(input_folder.glob("*.xlsx")) + list(input_folder.glob("*.xls")))
    cleaned_tables = []

    if not excel_files:
        raise FileNotFoundError(f"No Excel files found in {input_folder}")

    for file in excel_files:
        station_name = extract_station_name(file)
        df = pd.read_excel(file)
        df_clean = process_dgia_daily_file(df)

        station_matches = station_info.loc[station_info["station"] == station_name]
        if station_matches.empty:
            print(f"Skipping {file.name}: station metadata not found for '{station_name}'")
            continue

        station_row = station_matches.iloc[0]
        df_clean["id"] = station_row["code"]
        df_clean["name"] = station_name
        df_clean["latitude"] = station_row["lat"]
        df_clean["longitude"] = station_row["lon"]
        df_clean["altitude"] = station_row["altitude_m"]

        cleaned_tables.append(df_clean.reset_index())

    if not cleaned_tables:
        raise ValueError("No DGIA files could be processed.")

    df = pd.concat(cleaned_tables, ignore_index=True)
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df["month"] = df["time"].dt.month

    climatology = (
        df.groupby(["id", "month"], as_index=False)
        .agg(
            ssrd=("solar_radiation_W_m2", "mean"),
            name=("name", "first"),
            latitude=("latitude", "first"),
            longitude=("longitude", "first"),
            altitude=("altitude", "first"),
        )
    )

    # Convert W m-2 into kWh m-2 day-1, following the original workflow.
    climatology["ssrd"] = climatology["ssrd"] * w_m2_to_kwh_m2_day

    return standardise_output(climatology)

## Step 5. Source 2 processing functions

In [5]:
def process_monthly_summary_block(df_block, stations_info):
    """Process one station block from the DAVIS/PYTO monthly summary file."""
    df = df_block.copy()

    spanish_to_english_month = {
        "ENE": "Jan",
        "FEB": "Feb",
        "MAR": "Mar",
        "ABR": "Apr",
        "MAY": "May",
        "JUN": "Jun",
        "JUL": "Jul",
        "AGO": "Aug",
        "SET": "Sep",
        "OCT": "Oct",
        "NOV": "Nov",
        "DIC": "Dec",
    }

    month_number = {
        "Jan": 1,
        "Feb": 2,
        "Mar": 3,
        "Apr": 4,
        "May": 5,
        "Jun": 6,
        "Jul": 7,
        "Aug": 8,
        "Sep": 9,
        "Oct": 10,
        "Nov": 11,
        "Dec": 12,
    }

    stations = df.iloc[0]
    months = df.iloc[1]

    new_columns = []
    for station, month in zip(stations, months):
        if pd.isna(station) and pd.isna(month):
            new_columns.append(np.nan)
            continue

        if isinstance(month, str):
            month = spanish_to_english_month.get(month.strip(), month)

        if pd.notna(station) and pd.notna(month):
            new_columns.append(f"{station}_{month}")
        elif pd.notna(station):
            new_columns.append(station)
        else:
            new_columns.append(month)

    df.columns = new_columns
    df = df.loc[:, df.columns.notna()]
    df = df.iloc[3:].reset_index(drop=True)
    df = df.apply(pd.to_numeric, errors="ignore")

    station_name = df.columns[0]
    mean_rows = df[df[station_name] == "PROMEDIO"]
    if mean_rows.empty:
        return pd.DataFrame()

    row_mean = mean_rows.iloc[0]
    records = []

    for col in df.columns[1:]:
        if col not in month_number:
            continue

        value = row_mean[col]
        if pd.notna(value):
            records.append(
                {
                    "month": month_number[col],
                    "ssrd": value,
                    "name": station_name,
                }
            )

    out = pd.DataFrame(records)
    if out.empty:
        return out

    out["station_norm"] = out["name"].apply(normalize_name)
    stations_info = stations_info.copy()
    stations_info["station_norm"] = stations_info["name"].apply(normalize_name)

    out = out.merge(
        stations_info[["station_norm", "lat", "lon", "altitude"]],
        on="station_norm",
        how="left",
    )

    out = out.rename(columns={"lat": "latitude", "lon": "longitude"})
    out = out.drop(columns=["station_norm"])

    return out


def process_monthly_summary(summary_path, station_coordinates_path):
    """Process the full DAVIS/PYTO monthly summary file."""
    stations_info = pd.read_csv(station_coordinates_path)
    raw = pd.read_excel(summary_path)

    cleaned_blocks = []
    for start in np.arange(0, len(raw), 10):
        block = raw.iloc[start : start + 10]
        if len(block) < 4:
            continue

        cleaned = process_monthly_summary_block(block, stations_info)
        if not cleaned.empty:
            cleaned_blocks.append(cleaned)

    if not cleaned_blocks:
        raise ValueError("No monthly summary blocks could be processed.")

    df = pd.concat(cleaned_blocks, ignore_index=True)

    station_codes = {
        "San Camilo": 130700,
        "Palpa": 110698,
        "Miraflores": 120208,
        "La Pampilla": 120836,
        "La Joya": 130804,
        "Juli": 110880,
        "Incahuasi": None,
        "Huánuco": 120404,
        "Granja Kayra": 120607,
        "Ferreñafe": 110331,
        "Camaná": 110832,
        "Cajamarca": 130304,
    }

    df["id"] = df["name"].map(station_codes).astype("Int64")
    return standardise_output(df)

## Step 6. Run processing

In [6]:
tables = []

if source1_folder.exists() and station_info_path.exists():
    source1 = process_dgia_folder(source1_folder, station_info_path)
    tables.append(source1)
else:
    print("Source 1 skipped because the folder or station metadata file was not found.")

if source2_summary_path.exists() and source2_station_coordinates_path.exists():
    source2 = process_monthly_summary(source2_summary_path, source2_station_coordinates_path)
    tables.append(source2)
else:
    print("Source 2 skipped because the summary file or coordinate file was not found.")

if not tables:
    raise FileNotFoundError("No station source was available. Check the input paths in the User parameters cell.")

df_stats = pd.concat(tables, ignore_index=True)
df_stats = standardise_output(df_stats)

if excluded_station_names:
    df_stats = df_stats[~df_stats["name"].isin(excluded_station_names)].copy()

df_stats.to_csv(output_csv, index=False)

print(f"Saved: {output_csv}")
print(f"Rows: {len(df_stats)}")
print(f"Stations: {df_stats['name'].nunique()}")

Skipping HS_839_LA_PAMPILLA_.xls: station metadata not found for 'PAMPILLA'
Skipping RS_836_CHARACATO_PIRANOMETRO.xls: station metadata not found for ''
Skipping RS_HS_304_WEBERBAUER_PIRANOMETRO.xls: station metadata not found for 'WEBERBAUER'
Skipping RS_HS_489_COSMOS_PIRANOMETRO.xls: station metadata not found for 'COSMOS'
Skipping RS_HS_607_GJA_KAYRA_PIRANOMETRO.xls: station metadata not found for 'GJA_KAYRA'
Saved: /mnt/DATA/PROGETTI/27_WMO_ATLAS/stations/peru/allstats_solar_radiation.csv
Rows: 243
Stations: 21


/tmp/ipykernel_1376111/1213399973.py:57: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors="ignore")
/tmp/ipykernel_1376111/1213399973.py:57: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors="ignore")
/tmp/ipykernel_1376111/1213399973.py:57: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors="ignore")
/tmp/ipykernel_1376111/1213399973.py:57: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors="ignore")


## Step 7. Quick quality check

This optional cell shows the first rows of the final station table.

In [7]:
df_stats.head()

,id,month,ssrd,name,latitude,longitude,altitude
0,120362,1,4.606300,BAMBAMARCA,-6.6667,-78.5167,2536
1,120362,2,4.319925,BAMBAMARCA,-6.6667,-78.5167,2536
2,120362,3,4.547437,BAMBAMARCA,-6.6667,-78.5167,2536
3,120362,4,4.395945,BAMBAMARCA,-6.6667,-78.5167,2536
4,120362,5,4.447025,BAMBAMARCA,-6.6667,-78.5167,2536
